# RNN/LSTM 与序列状态

## 学习目标

能够解释 batch、time、feature 维度以及 LSTM 隐藏状态和单元状态。


## 概念模型与执行路径

RNN 在每个时间步复用参数并传递状态。LSTM 使用门控缓解长期依赖中的梯度问题。`batch_first=True` 只改变输入输出布局，不改变隐藏状态布局。


### 实验 1


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2


In [ ]:
import torch
from torch import nn
lstm = nn.LSTM(input_size=3, hidden_size=5, num_layers=2, batch_first=True)
sequence = torch.randn(4, 7, 3)
outputs, (hidden, cell) = lstm(sequence)
print("outputs:", outputs.shape)
print("hidden:", hidden.shape, "cell:", cell.shape)


### 实验 3


In [ ]:
torch.testing.assert_close(outputs[:, -1], hidden[-1])
print("last output equals top-layer final hidden state")


### 实验 4


In [ ]:
from examples.rnn_sequences import make_dataset
dataset = make_dataset(size=8, steps=12)
sample, label = dataset[0]
print("sample shape:", sample.shape, "direction label:", label.item())
print("first/last value:", sample[0].item(), sample[-1].item())


### 实验 5


In [ ]:
# python 07-deep-learning/pytorch/examples/rnn_sequences.py --quick --epochs 5


## 底层机制

隐藏状态 shape 是 `(layers * directions, batch, hidden)`。变长序列可使用 padding 和 packing，避免模型把补齐位置当作真实时间步。长序列仍可能需要截断反向传播。


## 检查点

若删除 `batch_first=True`，输入应从 `(batch, time, feature)` 改成什么？隐藏状态 shape 是否变化？


## 试一试

把 LSTM 改成双向，预测输出和隐藏状态 shape，再修改分类头输入维度。


## 常见错误与调试

混淆 batch/time、分类时取错层的 hidden、未 mask padding、跨 batch 保留状态却忘记 detach。
